## Prompt Chaining

In this example we want to make a workflow that make a blog from a given topic

Topic -> Generate Outline -> Generate Blog -> END

In [3]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict
from dotenv import load_dotenv
import os 

In [2]:
load_dotenv()

True

In [4]:
llm = ChatOpenAI(
    model="nvidia/nemotron-3-super-120b-a12b:free",
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
    timeout=30,
    max_retries=2,
)

In [5]:
class BlogState(TypedDict):
    title:str
    outline:str
    content:str

In [7]:
def create_outline(state: BlogState) -> BlogState:
    # Fetch the title
    title = state['title']

    # Call the LLM to generate output
    prompt = f'Generate a detailed outline for a blog on the topic: {title}'
    outline = llm.invoke(prompt).content

    # Update State
    state['outline'] = outline
    return state

In [9]:
def create_blog(state: BlogState) -> BlogState:
    title = state['title']
    outline = state['outline']

    prompt = f"Write a detailed blog on the title {title}, using the following outline \n {outline}"

    content = llm.invoke(prompt).content

    state['content'] = content

    return state

In [11]:
graph = StateGraph(BlogState)

graph.add_node('Create_Outline', create_outline)
graph.add_node('Create_Blog', create_blog)

graph.add_edge(START, 'Create_Outline')
graph.add_edge('Create_Outline', 'Create_Blog')
graph.add_edge('Create_Blog', END)

workflow = graph.compile()

In [12]:
intial_state = {'title' : 'Rise of AI in India'}

final_state = workflow.invoke(intial_state)

print(final_state)

KeyboardInterrupt: 